In [1]:
DATA_DIR = "../../data"

In [2]:
import pandas as pd

df = pd.read_csv(f"{DATA_DIR}/raw/results.csv")
df["date"] = pd.to_datetime(df["date"])

display(df)

,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,ct_1,t_2,t_1,ct_2,event_id,match_id,rank_1,rank_2,map_wins_1,map_wins_2,match_winner
0,0,2020-03-18,Recon 5,TeamOne,Dust2,0,16,2,2,0,1,0,15,5151,2340454,62,63,0,2,2
1,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,8,6,5,10,5151,2340454,62,63,0,2,2
2,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,9,6,3,10,5243,2340461,140,118,12,16,2
3,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,0,8,7,8,5151,2340453,61,38,0,2,2
4,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,4,5,4,11,5151,2340453,61,38,0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45768,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,8,7,5,9,1970,2299059,7,16,1,2,2
45769,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,10,5,6,8,1970,2299059,7,16,1,2,2
45770,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,7,8,9,4,1934,2299011,10,14,16,12,1
45771,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,4,1,12,3,1934,2299001,6,12,16,4,1


In [3]:
# We sort the datased by increasing date as ELO computation needs to be done chronologically
df = df.sort_values("date").reset_index(drop=True)

display(df)

,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,ct_1,t_2,t_1,ct_2,event_id,match_id,rank_1,rank_2,map_wins_1,map_wins_2,match_winner
0,45772,2015-11-03,NiP,Envy,Cobblestone,16,9,1,2,4,6,12,3,1934,2299003,6,1,16,9,1
1,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,4,1,12,3,1934,2299001,6,12,16,4,1
2,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,7,8,9,4,1934,2299011,10,14,16,12,1
3,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,10,5,6,8,1970,2299059,7,16,1,2,2
4,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,8,7,5,9,1970,2299059,7,16,1,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45768,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,4,5,4,11,5151,2340453,61,38,0,2,2
45769,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,0,8,7,8,5151,2340453,61,38,0,2,2
45770,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,9,6,3,10,5243,2340461,140,118,12,16,2
45771,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,8,6,5,10,5151,2340454,62,63,0,2,2


In [4]:
import sys

sys.path.append("../../src/shared")

from data_prep import (
    build_feature_state,
    compute_features_for_match,
    update_state_with_result,
)  # type: ignore

state = build_feature_state(df)
X = []
y = []

# Construct the features entirely from computed values, that's why X doesn't event concat the features with df
for _, match in df.iterrows():
    feats = compute_features_for_match(match, state)
    X.append(feats)
    y.append(int(match["match_winner"] == 1))
    state = update_state_with_result(match, state)

df_featured = pd.concat([df, pd.DataFrame(X), pd.Series(y, name="team_1_wins")], axis=1)

display(df_featured)

,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,ct_1,...,map_wins_1,map_wins_2,match_winner,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins
0,45772,2015-11-03,NiP,Envy,Cobblestone,16,9,1,2,4,...,16,9,1,0.000000,0.0,0.000000,0,5,0.5,1
1,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,4,...,16,4,1,16.000000,1.0,1.000000,1,-6,0.5,1
2,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,7,...,16,12,1,0.000000,0.0,0.000000,0,-4,0.5,1
3,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,10,...,1,2,2,0.000000,0.0,0.000000,0,-9,0.5,0
4,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,8,...,1,2,2,-32.000000,-1.0,-1.000000,0,-9,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45768,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,4,...,0,2,2,-78.253937,0.3,-0.166667,-18,23,0.0,0
45769,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,0,...,0,2,2,-103.165938,0.3,-0.166667,-18,23,0.0,0
45770,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,9,...,12,16,2,-37.342767,0.2,-0.157971,21,22,1.0,0
45771,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,8,...,0,2,2,39.282586,0.2,0.133333,-491,-1,0.5,0


In [5]:
df_featured.to_csv(f"{DATA_DIR}/featured/results.csv", index=False)